In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
import numpy as np
import json

2026-04-15 10:44:38.521785: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776249878.787266      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776249878.869835      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776249879.537279      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776249879.537325      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776249879.537329      16 computation_placer.cc:177] computation placer alr

In [2]:
train_gen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train = train_gen.flow_from_directory("/kaggle/input/datasets/leshray0211/dataset65/Comprehensive Disaster Dataset(CDD)/CDD_Augmented", target_size=(128,128),
                                      batch_size=32, class_mode='categorical', subset='training')

val = train_gen.flow_from_directory("/kaggle/input/datasets/leshray0211/dataset65/Comprehensive Disaster Dataset(CDD)/CDD_Augmented", target_size=(128,128),
                                    batch_size=32, class_mode='categorical', subset='validation')

Found 14644 images belonging to 6 classes.
Found 3659 images belonging to 6 classes.


In [3]:
print(train.class_indices)

{'Damaged_Infrastructure': 0, 'Fire_Disaster': 1, 'Human_Damage': 2, 'Land_Disaster': 3, 'Non_Damage': 4, 'Water_Disaster': 5}


In [4]:
model = Sequential([
    Conv2D(32,(3,3),activation='relu',input_shape=(128,128,3)),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128,activation='relu'),
    Dense(train.num_classes,activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-04-15 10:45:37.463556: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [6]:
model.fit(train, validation_data=val, epochs=3)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/3
458/458 ━━━━━━━━━━━━━━━━━━━━ 233s 506ms/step - accuracy: 0.6880 - loss: 1.1156 - val_accuracy: 0.7207 - val_loss: 0.7609
Epoch 2/3
458/458 ━━━━━━━━━━━━━━━━━━━━ 213s 464ms/step - accuracy: 0.7672 - loss: 0.6524 - val_accuracy: 0.7964 - val_loss: 0.5775
Epoch 3/3
458/458 ━━━━━━━━━━━━━━━━━━━━ 213s 465ms/step - accuracy: 0.8777 - loss: 0.3603 - val_accuracy: 0.8180 - val_loss: 0.5503


In [7]:
loss, acc = model.evaluate(val)

115/115 ━━━━━━━━━━━━━━━━━━━━ 20s 173ms/step - accuracy: 0.8219 - loss: 0.5424


In [8]:
img = load_img("/kaggle/input/datasets/leshray0211/dataset65/Comprehensive Disaster Dataset(CDD)/CDD_Augmented/Non_Damage/10004.jpg", target_size=(128,128))
print(img)
img = img_to_array(img)/255.0
img = np.expand_dims(img, axis=0)

<PIL.Image.Image image mode=RGB size=128x128 at 0x7800B894CCB0>


In [9]:
pred = model.predict(img)
pred_class = int(np.argmax(pred))
confidence = float(np.max(pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step


In [10]:
pred

array([[0.00320687, 0.06552334, 0.02967142, 0.00196321, 0.88780224,
        0.0118329 ]], dtype=float32)

In [11]:
def className(n):
    if n == 0:
        return "Damaged_Infrastrucutre"
    elif n == 1:
        return "Fire_Disaster"
    elif n == 2:
        return "Human_Damage"
    elif n == 3:
        return "Land_Disaster"
    elif n == 4:
        return "Non_Damage"
    else :
        return "Water_Disaster"

In [12]:
df = pd.DataFrame([{
    "prediction": className(pred_class),
    "confidence": confidence,
    "accuracy": acc
}])

NameError: name 'pd' is not defined

In [ ]:
df

In [ ]:
model.save('my_model.keras')

In [ ]:
def predict(img):
    img = load_img(img, target_size=(128,128))
    img = img_to_array(img)/255.0
    img = np.expand_dims(img, axis=0)
    pred = model.predict(img)
    pred_class = int(np.argmax(pred))
    return className(pred_class)

In [ ]:
import gradio as gr

In [ ]:
demo = gr.Interface(
    fn=predict, 
    inputs=gr.Image(type="filepath"), 
    outputs=gr.Label()
)
demo.launch()